# Notebook 06: Check the gene sets and score every cell

**Goal.** Confirm that the frozen gene sets are detected in your data, then compute one score per cell for each module (mitophagy, lysosome and four negative-control modules).

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** **Coverage check.** The two primary sets must not share genes (they do not). For each cohort the table shows what fraction of every gene set exists in the data. Below about 0.7 the module is unreliable in that cohort: say so in the Methods. Control modules are built from gene-name prefixes (Rpl/Rps = ribosome; Ndu/Sdh/Uqcr/Cox/Atp5 = oxidative phosphorylation).

*You should see:* All 1.0 in the simulation (real data are usually 0.85-1.0).

In [ ]:
from mlc.dataset import Dataset
from mlc import proc
modules = lambda ds: {"mitophagy": G.MITOPHAGY, "lysosome": G.LYSOSOME, "autophagy_core": G.AUTOPHAGY_CORE,
                      "proteasome": G.PROTEASOME, "ribosome": proc.prefix_genes(ds, G.RIBOSOME_PREFIX),
                      "oxphos": proc.prefix_genes(ds, G.OXPHOS_PREFIX)}
print("genes shared by mitophagy and lysosome sets:", set(G.MITOPHAGY) & set(G.LYSOSOME))
rows = []
for c in cohorts()["mouse"]:
    ds = Dataset.load(PROC / f"{c}_microglia")
    for name, genes in modules(ds).items():
        rows.append(dict(cohort=c, module=name, found=len(ds.gene_idx(genes)), total=len(genes)))
cov = pd.DataFrame(rows); cov["fraction"] = (cov.found / cov.total).round(2)
print(cov.pivot(index="module", columns="cohort", values="fraction"))

**Cell 3.** Computes **m_mitophagy, m_lysosome, m_ribosome ...** for every cell (set average minus matched-control average) and adds 0/1 columns for the cell states (used later to adjust for state). One column per module, one row per cell, saved as *{cohort}_scored*.

*You should see:* Six m_ columns per cohort.

In [ ]:
for c in cohorts()["mouse"]:
    ds = Dataset.load(PROC / f"{c}_microglia")
    proc.add_scores(ds, modules(ds), prefix="m_", seed=SEED)
    d = pd.get_dummies(ds.obs.state, prefix="state", drop_first=True).astype(float)
    for col in d: ds.obs[col] = d[col].values
    ds.save(PROC / f"{c}_scored")
    print(c, [x for x in ds.obs.columns if x.startswith("m_")])

**Cell 4.** A first look at the raw relationships in one mouse: the correlation of every module with every other. The mitophagy-lysosome correlation here is **not yet adjusted** for sequencing depth or cell state; notebook 07 does that. Do not interpret these numbers.

*You should see:* A 6 x 6 correlation table with ones on the diagonal.

In [ ]:
ds = Dataset.load(PROC / f"{cohorts()['mouse'][0]}_scored")
o = ds.obs[ds.obs.sample_id == ds.obs.sample_id.iloc[0]]
print(o[[c for c in o.columns if c.startswith("m_")]].corr(method="spearman").round(2))